In [2]:

import pandas as pd
from typing import List
from scipy.signal import butter, sosfiltfilt

from sklearn.utils.class_weight import compute_class_weight
import numpy as np
import torch
import xarray as xr

from sklearn.model_selection import GroupShuffleSplit
from sklearn.model_selection import ShuffleSplit


In [3]:
# PARAMS

DATA_PATH = 'data/LEMON_DATA/EC_all_channels_processed_downsampled.nc5'
DEMOGRAPHICS_PATH = 'data/LEMON_DATA/Demographics.csv'
PRETRAINED_CHECKPOINT_PATH = 'logs/06022025/06.02.2025_epoch_2500.model.keras'
CLS_CHECKPOINT_PATH = 'logs/20250221_classifier_v3'

CHANNELS = ['O1', 'O2', 'F1', 'F2', 'C1', 'C2', 'P1', 'P2']

DO_GROUPED_SHUFFLE = True
DO_DOWN_SAMPLE = True

In [ ]:

def load_data(data_path: str,
              channels: List[str],
              bandpass_filter: float = .5,
              time_dim: int = 512) -> tuple:

    xarray = xr.open_dataarray(data_path, engine='h5netcdf')
    n_subjects = xarray.subject.size

    demog = pd.read_csv(DEMOGRAPHICS_PATH, index_col="ID")

    # behavioral score    
    beh_score = pd.read_csv('data/LEMON_DATA/UPPS.csv', index_col="ID")
    beh_score_name = 'Gender_ 1=female_2=male'

    # merge behavioral score
    demog = demog.merge(beh_score, left_index=True, right_index=True)
    # -1 is to map 1-2 gender to 0-1
    beh_score = demog.loc[xarray["subject"].values, beh_score_name] - 1
    xarray = xarray.assign_coords(beh_score=("subject", beh_score))

    # age (young/old)
    # demog['is_old'] = demog['Age'].apply(lambda x:
    #     1 if int(x.split('-')[0]) >= 50 else 0)
    # beh_score = demog.loc[xarray["subject"].values, "is_old"]
    # xarray = xarray.assign_coords(beh_score=("subject", beh_score))

    if DO_DOWN_SAMPLE:
        n_y0 = (beh_score == 0).sum()
        n_y1 = (beh_score == 1).sum()
        n_min = min(n_y0, n_y1)
        n_subjects = n_min * 2
        y0_sub_ids = beh_score[beh_score == 0].index[:n_min]
        y1_sub_ids = beh_score[beh_score == 1].index[:n_min]
        sub_ids = y0_sub_ids.append(y1_sub_ids)
        xarray = xarray.sel(subject=sub_ids)

    x = xarray.sel(channel=channels).to_numpy()
    # n_subjects = x.shape[0]

    if bandpass_filter is not None:
        sos = butter(4, bandpass_filter, btype='high', fs=128, output='sos')  # TODO: fs
        x = sosfiltfilt(sos, x, axis=-1)

    x = torch.tensor(x.copy()).unfold(2, time_dim, time_dim).permute(0, 2, 3, 1).flatten(0, 1)  # TODO: copy was added because of an error, look into this

    sub = torch.tensor(np.arange(0, n_subjects).repeat(x.shape[0] // n_subjects)[:, np.newaxis])
    labels = xarray.beh_score.values

    y = labels.repeat(x.shape[0] / n_subjects)
    sub_ids_classifier = sub.squeeze().numpy()

    return x, y, sub_ids_classifier

X, y, groups = load_data(DATA_PATH, CHANNELS)

class_weights = compute_class_weight('balanced', classes=np.unique(y), y=y)
class_weights = {'0': class_weights[0], '1': class_weights[1]}

n_subjects = len(np.unique(groups))
print(X.shape, y.shape, n_subjects)

torch.Size([13468, 512, 8]) (13468,) 148


In [42]:
# SPLIT

if DO_GROUPED_SHUFFLE:
    group_shuffle = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=1)
    train_idx, val_idx = next(group_shuffle.split(X, y, groups=groups))
else:
    random_shuffle = ShuffleSplit(n_splits=1, test_size=0.2, random_state=1)
    train_idx, val_idx = next(random_shuffle.split(X, y))

# report ratio of classes in train and val
y.mean(), y[train_idx].mean(), y[val_idx].mean()

(np.float64(0.5),
 np.float64(0.5169491525423728),
 np.float64(0.43333333333333335))

In [43]:
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.pipeline import Pipeline
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier

pipe = Pipeline([
    ('scaler', StandardScaler()),
    ('pca', PCA(n_components=0.99)),
    # ('clf', SVC(kernel='linear', verbose=True))
    ('clf', RandomForestClassifier(verbose=2, n_jobs=-1))
])

# fit the model
pipe.fit(torch.flatten(X[train_idx], 1), y[train_idx])
# predict the response
y_pred = pipe.predict(torch.flatten(X[val_idx], 1))
# evaluate accuracy
from sklearn.metrics import accuracy_score
print("Accuracy: ", accuracy_score(y[val_idx], y_pred))


[Parallel(n_jobs=-1)]: Using backend ThreadingBackend with 12 concurrent workers.


building tree 1 of 100building tree 2 of 100
building tree 3 of 100
building tree 7 of 100
building tree 10 of 100
building tree 12 of 100
building tree 8 of 100
building tree 11 of 100
building tree 5 of 100
building tree 4 of 100
building tree 6 of 100
building tree 9 of 100

building tree 13 of 100
building tree 14 of 100
building tree 15 of 100
building tree 16 of 100
building tree 17 of 100
building tree 18 of 100
building tree 19 of 100
building tree 20 of 100
building tree 21 of 100
building tree 22 of 100
building tree 23 of 100
building tree 24 of 100
building tree 25 of 100
building tree 26 of 100
building tree 27 of 100
building tree 28 of 100
building tree 29 of 100
building tree 30 of 100
building tree 31 of 100
building tree 32 of 100
building tree 33 of 100
building tree 34 of 100
building tree 35 of 100
building tree 36 of 100


[Parallel(n_jobs=-1)]: Done  17 tasks      | elapsed:    1.1s


building tree 37 of 100
building tree 38 of 100
building tree 39 of 100
building tree 40 of 100
building tree 41 of 100
building tree 42 of 100
building tree 43 of 100
building tree 44 of 100
building tree 45 of 100
building tree 46 of 100
building tree 47 of 100
building tree 48 of 100
building tree 49 of 100
building tree 50 of 100
building tree 51 of 100
building tree 52 of 100
building tree 53 of 100
building tree 54 of 100
building tree 55 of 100
building tree 56 of 100
building tree 57 of 100
building tree 58 of 100
building tree 59 of 100
building tree 60 of 100
building tree 61 of 100
building tree 62 of 100
building tree 63 of 100
building tree 64 of 100
building tree 65 of 100
building tree 66 of 100
building tree 67 of 100
building tree 68 of 100
building tree 69 of 100
building tree 70 of 100
building tree 71 of 100
building tree 72 of 100
building tree 73 of 100
building tree 74 of 100
building tree 75 of 100
building tree 76 of 100
building tree 77 of 100
building tree 78

[Parallel(n_jobs=-1)]: Done 100 out of 100 | elapsed:    5.0s finished
[Parallel(n_jobs=12)]: Using backend ThreadingBackend with 12 concurrent workers.
[Parallel(n_jobs=12)]: Done  17 tasks      | elapsed:    0.0s
[Parallel(n_jobs=12)]: Done 100 out of 100 | elapsed:    0.0s finished


Accuracy:  0.5945054945054945
